In [2]:
import requests

def fetch_clinicaltrials_articles_with_metadata(query: str, max_results=3, use_mock_if_empty=True):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Step 1: Search ClinicalTrials.gov
    search_url = "https://clinicaltrials.gov/api/v2/studies"
    search_params = {
        "query.term": query,
        "pageSize": max_results,
        "format": "json"
    }

    try:
        search_response = requests.get(
            search_url,
            params=search_params,
            headers=headers,
            timeout=10
        )
        search_response.raise_for_status()
        search_data = search_response.json()

        studies = search_data.get("studies", [])
        print("Found ClinicalTrials studies:", len(studies))

        if not studies:
            raise ValueError("No studies found for this query.")

        articles_info = []

        # Step 2: Extract study summaries
        for study in studies:
            protocol_section = study.get("protocolSection", {})
            identification_module = protocol_section.get("identificationModule", {})
            description_module = protocol_section.get("descriptionModule", {})
            status_module = protocol_section.get("statusModule", {})
            conditions_module = protocol_section.get("conditionsModule", {})
            sponsor_module = protocol_section.get("sponsorCollaboratorsModule", {})

            # NCT ID
            nct_id = identification_module.get("nctId", "No NCT ID")

            # Title
            title = identification_module.get("briefTitle", "No title")

            # Abstract / Summary
            abstract = description_module.get("briefSummary", "No summary available")

            # Authors equivalent -> Sponsor / Collaborators
            sponsor_name = "No sponsor listed"
            lead_sponsor = sponsor_module.get("leadSponsor", {})
            if isinstance(lead_sponsor, dict):
                sponsor_name = lead_sponsor.get("name", "No sponsor listed")

            authors = [sponsor_name]

            # Publication date equivalent -> Start date
            pub_date = "No date"
            start_date_struct = status_module.get("startDateStruct", {})
            if isinstance(start_date_struct, dict):
                pub_date = start_date_struct.get("date", "No date")

            # Condition
            conditions = conditions_module.get("conditions", [])
            if not conditions:
                conditions = ["No condition listed"]

            # Trial URL
            if nct_id != "No NCT ID":
                url = f"https://clinicaltrials.gov/study/{nct_id}"
            else:
                url = "No URL available"

            print(
                f"Article: {title}\n"
                f"   - Authors: {authors}\n"
                f"   - Date: {pub_date}\n"
                f"   - Conditions: {conditions}\n"
                f"   - URL: {url}\n"
            )

            articles_info.append({
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "publication_date": pub_date,
                "conditions": conditions,
                "article_url": url
            })

        if not articles_info and use_mock_if_empty:
            print("No valid studies found, returning mock data.")
            return [{
                "title": "Simulated Clinical Trial on Fever",
                "abstract": "This is a simulated summary of a clinical trial on fever treatment.",
                "authors": ["Mock Sponsor"],
                "publication_date": "2024-03-01",
                "conditions": ["Fever"],
                "article_url": "https://clinicaltrials.gov/study/NCT00000000"
            }]

        return articles_info

    except Exception as e:
        print(f"Error during ClinicalTrials.gov fetch: {e}")
        if use_mock_if_empty:
            return [{
                "title": "Simulated Clinical Trial on Fever",
                "abstract": "This is a simulated summary of a clinical trial on fever treatment.",
                "authors": ["Mock Sponsor"],
                "publication_date": "2024-03-01",
                "conditions": ["Fever"],
                "article_url": "https://clinicaltrials.gov/study/NCT00000000"
            }]
        else:
            return [{"message": f"Error: {e}"}]

In [3]:
fetch_clinicaltrials_articles_with_metadata("back pain")

Found ClinicalTrials studies: 3
Article: Training Therapy for the Prevention of Back Pain
   - Authors: ['Medical University of Vienna']
   - Date: 2017-11-01
   - Conditions: ['Back Pain', 'Lower Back Pain']
   - URL: https://clinicaltrials.gov/study/NCT03393429

Article: Neck Mobs and Impingement
   - Authors: ['Walsh University']
   - Date: 2011-12
   - Conditions: ['Shoulder and Neck']
   - URL: https://clinicaltrials.gov/study/NCT01744002

Article: Predictive Factors for the Success of Rehabilitation Programs in Chronic Low Back Pain
   - Authors: ['Centre Hospitalier Universitaire de Nīmes']
   - Date: 2025-02-19
   - Conditions: ['Low Back Pain']
   - URL: https://clinicaltrials.gov/study/NCT07051772



[{'title': 'Training Therapy for the Prevention of Back Pain',
  'abstract': 'Work place related (lower) back pain in medical personnel is limiting to workability.\n\nEven though occupational prevention programs are increasingly established, data on the effectiveness of training interventions offered at work-sites is largely missing.\n\nIn this randomized, investigator-blind, controlled feasibility study we aim to compare the effectiveness of device assisted training therapy in comparison to a general recommendation "to stay active" or group gymnastics in terms of pain frequency and intensity (main outcome). Additional outcome variables are: quality of life, psychological well-being, work efficiency, of sick-leave days.\n\nEligible employees (2 x 30) of the General Hospital of Vienna (AKH) over the age of 45 years suffering from (lower) back pain (\\>30 days/last year) of intensity ≥ 3 (numeric scale 0-10) will be included in two parallel groups. Group I starts with a device (DAVID) as

In [ ]:
print("hello bro")

hello
